In [0]:
%pip install mlflow==2.10.1 langchain==0.1.5 databricks-vectorsearch==0.22 databricks-sdk==0.18.0 mlflow[databricks]
dbutils.library.restartPython()

In [0]:
import os
from langchain_community.embeddings import EndpointEmbeddingWrapper
from langchain.chains import RetrievalQA
from langchain.llms import DatabricksModel
from databricks.vector_search import DatabricksVectorSearch

In [0]:
# Configuración
DATABRICKS_HOST = "https://https://dbc-ad7d5e59-0280.cloud.databricks.com/"  # <- Pon aquí tu workspace
DATABRICKS_TOKEN = ""         # <- O define el token directamente
INDEX_NAME = "bluetab.rag.idx"
EMBEDDING_ENDPOINT = "simple_embedding"                   # <- Nombre de tu endpoint de embeddings
LLM_ENDPOINT = "flan_t5_base_model"                      # <- Tu endpoint de LLM en Model Serving

In [0]:
# 1. Embedding model para las queries (wrapper al endpoint)
embedding_model = EndpointEmbeddingWrapper(
    endpoint_url=f"{DATABRICKS_HOST}/serving-endpoints/{EMBEDDING_ENDPOINT}/invocations",
    token=DATABRICKS_TOKEN
)

In [0]:
# 2. Vector Search retriever (usa embeddings ya precalculados en el índice)
retriever = DatabricksVectorSearch(
    index_name=INDEX_NAME,
    embedding=embedding_model
).as_retriever()

In [0]:

# 3. LLM desde tu endpoint en Model Serving
llm = DatabricksModel(
    endpoint_name=LLM_ENDPOINT,
    databricks_host=DATABRICKS_HOST,
    databricks_token=DATABRICKS_TOKEN
)

In [0]:





# 4. Cadena RAG (Retriever + Generador)
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

# 5. Ejemplo de consulta
query = "¿Qué modelos open-source puedo usar con Databricks Vector Search?"

respuesta = rag_chain(query)

# Resultado
print("Respuesta:")
print(respuesta["result"])

print("\nDocumentos usados como contexto:")
for doc in respuesta["source_documents"]:
    print("-", doc.metadata.get("id", "[sin id]"), ":", doc.page_content[:150], "...")
